# 02 — Skill roster
**Step 2 of 3.** Build the candidate universe, score each wallet from its
**closed-position history**, apply the hard gates (Blueprint §3), and save a
vetted roster to `roster.json`. Run this **weekly**.

This is our main defense against survivorship/luck bias: a wallet earns its place
by a *long, profitable, consistent* track record — not by being hot this month.


In [1]:
import importlib, pmc, json, time
from datetime import datetime, timezone, timedelta
importlib.reload(pmc)
from pmc import CFG, get_leaderboard, get_closed_positions, get_open_positions, portfolio_value
import pandas as pd

## 1. Candidate universe — union of `all` (proven) and `month` (active)

In [2]:
candidates = {}
for w in CFG.LEADERBOARD_WINDOWS:
    for row in get_leaderboard(window=w, limit=CFG.LEADERBOARD_TOP_N):
        candidates.setdefault(row["wallet"], row)
print(f"{len(candidates)} unique candidate wallets")

256 unique candidate wallets


## 2. Score each candidate from closed positions
For every wallet we compute the skill-gate metrics. Each resolved position is a
'trade'; `realizedPnl > 0` counts as a win. ROI = total realized PnL / total cost.

In [3]:
def score_wallet(wallet: str) -> dict | None:
    closed = get_closed_positions(wallet)
    if not closed:
        return None
    pnls, costs, last_ts = [], [], 0
    for p in closed:
        rp = float(p.get("realizedPnl") or p.get("cashPnl") or 0)
        cost = float(p.get("totalBought") or p.get("initialValue") or 0)
        pnls.append(rp); costs.append(abs(cost))
        ts = p.get("endDate") or p.get("timestamp") or 0
        if isinstance(ts, (int, float)):
            last_ts = max(last_ts, ts)
    n = len(pnls)
    total_pnl = sum(pnls)
    total_cost = sum(costs) or 1.0
    wins = sum(1 for x in pnls if x > 0)
    win_rate = wins / n
    roi = total_pnl / total_cost
    best_share = (max(pnls) / total_pnl) if total_pnl > 0 else 1.0
    return {
        "wallet": wallet,
        "resolved_trades": n,
        "lifetime_pnl": round(total_pnl, 2),
        "win_rate": round(win_rate, 3),
        "roi": round(roi, 3),
        "best_trade_share": round(best_share, 3),
        "n_positions": n,
    }

scored = []
for i, w in enumerate(candidates):
    s = score_wallet(w)
    if s:
        # use the leaderboard's authoritative lifetime PnL (our sample is capped/recent)
        s["lifetime_pnl"] = round(float(candidates[w].get("pnl") or s["lifetime_pnl"]), 2)
        scored.append(s)
    if (i+1) % 20 == 0: print(f"  scored {i+1}/{len(candidates)}")
sdf = pd.DataFrame(scored)
print(f"scored {len(sdf)} wallets")
sdf.head()

  scored 20/256
  scored 40/256
  scored 60/256
  scored 80/256
  scored 100/256
  scored 120/256
  scored 140/256
  scored 160/256
  scored 180/256
  scored 200/256
  scored 220/256
  scored 240/256
scored 255 wallets


,wallet,resolved_trades,lifetime_pnl,win_rate,roi,best_trade_share,n_positions
0,0x56687bf447db6ffa42ffe2204a05edaa20f55839,22,22053933.75,0.818,0.513,0.376,22
1,0x1f2dd6d473f3e824cd2f8a89d9c69fb96f6ad0cf,66,16619506.63,0.606,0.219,0.584,66
2,0x204f72f35326db932158cba6adff0b9a1da95e14,300,14194136.51,0.203,-0.011,1.000,300
3,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,300,11386689.78,0.547,0.087,0.250,300
4,0x2005d16a84ceefa912d4e380cd32e7ff827875ea,300,10352420.91,0.000,0.000,1.000,300


## 3. Apply the hard gates (Blueprint §3)

In [4]:
def passes_gate(r) -> bool:
    return (
        r["resolved_trades"] >= CFG.MIN_RESOLVED_TRADES and
        r["lifetime_pnl"]   >  CFG.MIN_LIFETIME_PNL and
        r["win_rate"]       >= CFG.MIN_WIN_RATE and
        r["roi"]            >= CFG.MIN_ROI and
        r["best_trade_share"] <= CFG.MAX_SINGLE_TRADE_PROFIT_SHARE and
        r["n_positions"]    <= CFG.MARKET_MAKER_POSITION_COUNT   # exclude market-makers
    )

roster = sdf[sdf.apply(passes_gate, axis=1)].copy() if len(sdf) else sdf
# skill score: normalize a blend of win_rate, roi, log(pnl) to 0..1
if len(roster):
    import numpy as np
    z = lambda s: (s - s.min()) / ((s.max() - s.min()) or 1)
    roster["skill"] = (0.4*z(roster["win_rate"]) + 0.4*z(roster["roi"]) +
                       0.2*z(np.log1p(roster["lifetime_pnl"].clip(lower=1)))).round(3)
    roster = roster.sort_values("skill", ascending=False).head(CFG.TARGET_ROSTER_SIZE)
print(f"roster size: {len(roster)}")
roster

roster size: 29


,wallet,resolved_trades,lifetime_pnl,win_rate,roi,best_trade_share,n_positions,skill
49,0xd38b71f3e8ed1af71983e5c309eac3dfa9b35029,287,2673262.16,0.976,0.513,0.035,287,0.965
28,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,300,4055583.63,0.950,0.458,0.061,300,0.918
40,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,97,3133968.20,0.959,0.248,0.228,97,0.733
135,0x9b3dcd99eec7fe11602e6534e6302c0f318d7422,110,1069068.64,0.891,0.403,0.043,110,0.720
25,0xdc876e6873772d38716fda7f2452a78d426d7ab6,300,4526176.05,0.717,0.321,0.277,300,0.562
147,0x3de4543d599ffb09386aac2eab198a295511b032,106,981995.24,0.660,0.477,0.097,106,0.532
59,0x7fb7ad0d194d7123e711e7db6c9d418fac14e33d,300,2282134.46,0.733,0.279,0.366,300,0.498
94,0xde7be6d489bce070a959e0cb813128ae659b5f4b,190,1537305.38,0.774,0.255,0.304,190,0.495
43,0x5966db1fe50763c9e3c014d756369bad07e1f804,115,3044108.69,0.730,0.239,0.089,115,0.480
80,0xea2b4224411e723499a803ce3f4758779fb31fc6,300,1777295.69,0.673,0.269,0.092,300,0.409


## 4. Save the roster (consumed by notebook 03)

In [5]:
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "config": {k: getattr(CFG, k) for k in ["MIN_RESOLVED_TRADES","MIN_LIFETIME_PNL","MIN_WIN_RATE","MIN_ROI"]},
    "wallets": roster.to_dict("records") if len(roster) else [],
}
with open("roster.json", "w") as f:
    json.dump(payload, f, indent=2, default=str)
print(f"saved roster.json with {len(payload['wallets'])} wallets")

saved roster.json with 29 wallets


---
**Tuning note.** If the roster comes back empty, your gates are too strict for the
sample the API returned — loosen `MIN_RESOLVED_TRADES` / `MIN_LIFETIME_PNL` in
`pmc.py` and re-run. The backtest (future notebook) is what sets these properly.

**Next:** `03_consensus_signals.ipynb`.